# Driver monitoring: local data, optional training

Run the numbered cells in order. The default checks your local prepared data and saved subject splits; **it does not train, download data, install packages, or change any Kaggle session**.

Choose either **State Farm distraction** or **RT-BENE eye state**. These are separate per-frame classifiers. Gaze off road, yawning and head nodding need suitable labels and are not trained here. DMD stays held out for later testing and never enters training.

Your notebook and source code live in this repository. A remote kernel runs on its own machine: it needs an explicit copy of both source code and prepared data. See `docs/LOCAL_NOTEBOOK.md` for preparation and manual transfer.


## 1. Choose the task and check where Python is running

Leave `RUN_TRAINING = False` for inspection. Selecting a remote kernel does not move files from your laptop. For Kaggle, attach a readable source snapshot and a prepared-data bundle first, then fill in the two remote paths below. The prepared bundle must contain `data/state_farm/prepared`, `data/rt_bene/prepared` (as needed), and `data/splits`.

`FAST_RUN` is only a small training smoke run when training is enabled; it is not a publishable evaluation.


In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import json
import os
import platform
import subprocess
import sys

RUN_TRAINING = False
TASK = "state_farm"  # "state_farm" or "rt_bene"
DISTRACTION_MODEL = "efficientnet_b2"  # Then compare "resnet18" with this same code upload.
FAST_RUN = False
DEVICE = "cuda"  # Set to "cuda" only with a GPU kernel.
LOCAL_REPO_ROOT = ""  # Usually blank: find this repo from the notebook directory.
REMOTE_REPO_ROOT = ""  # Auto-detect the current uploaded model code snapshot.
REMOTE_BUNDLE_ROOT = "/kaggle/input/datasets/mfahadwaqar2002/dlcv-prepared-training"
RESULTS_RUN = ""  # Optional existing experiment folder to inspect in step 6.

ON_KAGGLE = Path("/kaggle/working").is_dir()
print("Kernel:", "Kaggle remote" if ON_KAGGLE else "local / other kernel")
print("Python:", sys.executable)
print("Version:", platform.python_version())
print("Working directory:", Path.cwd())
print("Task:", TASK, "| Training enabled:", RUN_TRAINING, "| Device:", DEVICE)
assert TASK in {"state_farm", "rt_bene"}, "Choose one of the two supported tasks."
assert DEVICE in {"cpu", "cuda"}
assert DISTRACTION_MODEL in {"efficientnet_b2", "resnet18", "efficientnet_b0", "mobilenet_v3_small"}
if ON_KAGGLE:
    print("This kernel cannot read your laptop's E: drive. Remote copies are required.")


## 2. Locate code and prepared data

This only reads paths and checks installed packages. It never clones a repository or downloads raw data. Local setup instructions are in the companion guide. On a different remote service, explicitly set `LOCAL_REPO_ROOT` to the source on that machine; automatic Kaggle detection only applies to `/kaggle` kernels.


In [ ]:
def find_kaggle_source(input_root):
    """Check only the named code input and one archive wrapper level."""
    datasets = [input_root / "dlcv-project-code"]
    datasets.extend(input_root.glob("datasets/*/dlcv-project-code"))
    candidates = []
    for dataset in datasets:
        candidates.append(dataset / "project")
        candidates.extend(dataset.glob("*/project"))
    matches = sorted({p.resolve() for p in candidates if
        (p / "src/train/phase1.py").is_file()
        and (p / "src/train/notebook_backup.py").is_file()
        and (p / "configs/state_farm_efficientnet_b2.yaml").is_file()
    })
    if len(matches) > 1:
        raise ValueError(
            "Multiple current model source snapshots found. Set REMOTE_REPO_ROOT explicitly: "
            + str([str(p) for p in matches])
        )
    return matches[0] if matches else None


if ON_KAGGLE:
    REPO = (Path(REMOTE_REPO_ROOT).resolve() if REMOTE_REPO_ROOT
            else find_kaggle_source(Path("/kaggle/input")))
    BUNDLE = Path(REMOTE_BUNDLE_ROOT).resolve() if REMOTE_BUNDLE_ROOT else None
else:
    candidates = (
        [Path(LOCAL_REPO_ROOT).resolve()] if LOCAL_REPO_ROOT else [Path.cwd(), *Path.cwd().parents]
    )
    REPO = next((p for p in candidates if (p / "src/train/phase1.py").is_file()), None)
    BUNDLE = REPO

REPO_READY = REPO is not None and (REPO / "src/train/phase1.py").is_file()
DATA_BASE = BUNDLE / "data" if BUNDLE else None
MODEL_CONFIGS = {
    "efficientnet_b2": "configs/state_farm_efficientnet_b2.yaml",
    "resnet18": "configs/state_farm_resnet18.yaml",
    "efficientnet_b0": "configs/state_farm_efficientnet.yaml",
    "mobilenet_v3_small": "configs/state_farm_pretrained.yaml",
}
SPECS = {
    "state_farm": {
        "config": MODEL_CONFIGS[DISTRACTION_MODEL],
        "label": "label_distraction",
        "classes": [0, 1, 2, 3, 5],
    },
    "rt_bene": {"config": "configs/phase1.yaml", "label": "label_drowsiness", "classes": [0, 1]},
}
TRAIN_PACKAGES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
}
missing_packages = [
    name for module, name in TRAIN_PACKAGES.items() if importlib.util.find_spec(module) is None
]
print("Source root:", REPO or "MISSING: set a source path")
print("Prepared bundle root:", BUNDLE or "MISSING: set a bundle path")
print("Missing training packages:", missing_packages or "none")
if ON_KAGGLE and DEVICE == "cuda" and "torch" not in missing_packages:
    import torch
    assert torch.cuda.is_available(), "Enable GPU in Kaggle and reconnect this notebook."
    print("GPU:", torch.cuda.get_device_name(0))
if REPO_READY:
    # Prefer this explicit source snapshot over any editable installation in the kernel.
    if str(REPO) in sys.path:
        sys.path.remove(str(REPO))
    sys.path.insert(0, str(REPO))
    loaded_src = sys.modules.get("src")
    if loaded_src is not None and Path(loaded_src.__file__).resolve() != REPO / "src/__init__.py":
        raise RuntimeError(
            "Another source snapshot is already imported. Restart the kernel, then rerun."
        )
else:
    print("BLOCKED: source code is not available in this kernel.")
if DATA_BASE:
    for task in SPECS:
        print(task, "prepared root:", DATA_BASE / task / "prepared")
        print(task, "saved split:", DATA_BASE / "splits" / f"{task}_splits.json")


## 3. Check manifests, every image path, and fixed subject splits

Preparation happens explicitly in the terminal (`python scripts/prepare_local_data.py`; see the guide). This cell reads each prepared `manifest.csv`, completed preparation report and saved split JSON. Missing data are reported as blocked. Existing splits must match the manifest, seed and fractions exactly; this cell never creates or reshuffles them. All images are checked for existence; this is not a full image-decode audit.

RT-BENE uses labeled left-eye crops: **0 = open, 1 = closed/blinking**. State Farm uses the existing five project classes. Class mappings retain the limitations described in the dataset documentation.


In [ ]:
from collections import Counter, defaultdict
from pathlib import PurePosixPath
from time import perf_counter


def check_prepared_images(root, rows, task):
    """Check every referenced file, resolving each parent directory only once."""
    root = root.resolve()
    groups = defaultdict(list)
    for row in rows:
        raw = row["path"]
        relative = PurePosixPath(raw)
        if (
            not relative.name
            or relative.is_absolute()
            or ".." in relative.parts
            or "\\" in raw
            or any(":" in part for part in relative.parts)
        ):
            raise ValueError(f"Invalid relative image path: {raw}")
        groups[relative.parent].append(relative.name)
    checked, missing = 0, []
    started = perf_counter()
    print(f"{task}: checking {len(rows):,} image paths in {len(groups)} directories...", flush=True)
    for folder, names in groups.items():
        directory = (root / folder).resolve()
        if not directory.is_relative_to(root):
            raise ValueError(f"Image directory escapes prepared root: {folder}")
        expected = set(names)
        present = set()
        try:
            with os.scandir(directory) as entries:
                for entry in entries:
                    if entry.name not in expected:
                        continue
                    if entry.is_symlink():
                        target = Path(entry.path).resolve()
                        if not target.is_relative_to(root):
                            raise ValueError(f"Image symlink escapes prepared root: {entry.name}")
                        if target.is_file():
                            present.add(entry.name)
                    elif entry.is_file(follow_symlinks=False):
                        present.add(entry.name)
        except (FileNotFoundError, NotADirectoryError):
            pass
        missing.extend((folder / name).as_posix() for name in names if name not in present)
        checked += len(names)
        print(
            f"{task}: {checked:,}/{len(rows):,} paths checked "
            f"({perf_counter() - started:.1f}s)",
            flush=True,
        )
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing; first: {missing[0]}")


readiness = {}
loaded = {}
if REPO_READY and importlib.util.find_spec("yaml") is not None:
    import yaml
    from src.data.manifest import load_manifest
    from src.data.splits import assert_no_leakage, fingerprint
    import src.data.manifest as manifest_module

    assert Path(manifest_module.__file__).resolve() == REPO / "src/data/manifest.py"

    for task, spec in SPECS.items():
        try:
            print(f"{task}: reading manifest and preparation report...", flush=True)
            if DATA_BASE is None:
                raise FileNotFoundError("Prepared bundle root is not set")
            root = (DATA_BASE / task / "prepared").resolve()
            manifest = root / "manifest.csv"
            split_path = DATA_BASE / "splits" / f"{task}_splits.json"
            config_path = REPO / spec["config"]
            config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
            assert config.get("label_key", "label_drowsiness") == spec["label"]
            assert config["class_ids"] == spec["classes"], "Unexpected task class configuration"
            rows = load_manifest(manifest)
            report_path = DATA_BASE / task / "preparation_report.json"
            preparation = json.loads(report_path.read_text(encoding="utf-8"))
            assert preparation["dataset"] == task, "Preparation report dataset differs"
            assert preparation["images_decoded"] == len(rows), "Preparation decode count differs"
            assert preparation["missing_images"] == 0, "Preparation reported missing images"
            assert preparation["corrupt_images"] == 0, "Preparation reported corrupt images"
            assert (
                preparation["split_provenance"][
                    "exact_sample_labels_and_subject_memberships_preserved"
                ]
                is True
            ), "Historical split preservation not verified"
            assert {r["dataset"] for r in rows} == {task}, "Unexpected dataset; DMD is forbidden"
            check_prepared_images(root, rows, task)
            print(f"{task}: verifying fixed splits and class support...", flush=True)
            saved = json.loads(split_path.read_text(encoding="utf-8"))
            assert saved["fingerprint"] == fingerprint(rows), "Manifest differs from saved split"
            assert saved["seed"] == config["seed"], "Saved split seed differs from config"
            assert saved["fractions"] == config["split_fractions"], "Split fractions differ"
            parts = saved["splits"]
            assert set(parts) == {"train", "val", "test"}
            assert_no_leakage(parts)
            assert fingerprint([r for group in parts.values() for r in group]) == fingerprint(
                rows
            ), "Saved split rows differ"
            for name, group in parts.items():
                support = {r[spec["label"]] for r in group} - {-1}
                assert group and support, f"Empty or unlabeled {name} split"
                assert support <= set(spec["classes"]), f"Unsupported labels in {name}"
                if name in {"train", "val"}:
                    assert support == set(spec["classes"]), f"Missing class support in {name}"
            loaded[task] = {
                "root": root,
                "manifest": manifest,
                "split_path": split_path,
                "config_path": config_path,
                "config": config,
                "parts": parts,
                "fingerprint": saved["fingerprint"],
            }
            readiness[task] = {
                "ready": True,
                "images": len(rows),
                "subjects": len({r["subject_id"] for r in rows}),
                "split_fingerprint": saved["fingerprint"],
                "splits": {
                    name: {
                        "images": len(group),
                        "subjects": len({r["subject_id"] for r in group}),
                        "labels": dict(Counter(r[spec["label"]] for r in group)),
                    }
                    for name, group in parts.items()
                },
            }
            print(f"{task}: READY", flush=True)
        except Exception as exc:
            readiness[task] = {"ready": False, "reason": f"{type(exc).__name__}: {exc}"}
else:
    readiness = {
        task: {"ready": False, "reason": "Source root or PyYAML missing (step 2)"} for task in SPECS
    }
print(json.dumps(readiness, indent=2))
print("Selected task:", "READY" if readiness[TASK]["ready"] else "BLOCKED")


## 4. Read the model-comparison plan

The existing EfficientNet-B0 reference achieved 86.50% validation accuracy and 81.18% macro-F1. Next compare `efficientnet_b2` (288 pixels, batch 16) and `resnet18` (224 pixels, batch 32), one run at a time using the same code upload. Both use pretrained weights, Adam at 0.0001 for 10 epochs, unchanged classes/driver splits, and no test scoring. B2 changes model size, image resolution and batch size together.

Choose `DISTRACTION_MODEL` in step 1. Use `RUN_TRAINING = False` for inspection first. Step 4 must show the selected architecture and config. These runs use only State Farm training images; RT-BENE is a separate eye-state task. NITYMED annotations remain incomplete, and DMD is held out.


In [ ]:
import hashlib


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


source_files = {}
source_revision = None
source_dirty = None
if REPO_READY:
    paths = sorted(
        set(REPO.glob("src/**/*.py")) | set(REPO.glob("configs/*.yaml")) | {REPO / "pyproject.toml"}
    )
    source_files = {p.relative_to(REPO).as_posix(): sha256_file(p) for p in paths if p.is_file()}
    try:
        source_revision = subprocess.check_output(
            ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
        source_dirty = bool(
            subprocess.check_output(
                ["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"],
                text=True,
                stderr=subprocess.DEVNULL,
            ).strip()
        )
    except (OSError, subprocess.CalledProcessError):
        marker = REPO / "SOURCE_COMMIT"
        source_revision = (
            marker.read_text(encoding="utf-8").strip()
            if marker.is_file()
            else "snapshot-without-git"
        )
    print("Source revision:", source_revision, "| Tracked changes:", source_dirty)
    print("Source/config files hashed:", len(source_files))

if readiness[TASK]["ready"]:
    selected = loaded[TASK]
    print("Config:", selected["config_path"])
    print(json.dumps(selected["config"], indent=2))
    print("Fixed split:", selected["split_path"])
    print("Device:", DEVICE, "| Fast smoke run:", FAST_RUN)
    print(
        "Outputs parent:",
        Path("/kaggle/working/experiments") if ON_KAGGLE else REPO / "results/experiments",
    )
else:
    print("Resolve the selected task's blocked status before enabling training.")


## 5. Optional training

To train intentionally, change `RUN_TRAINING` in step 1, choose the task/device, then rerun steps 1–5. Local CPU training may take a long time. GPU training requires an available CUDA runtime in the selected kernel.

Every invocation creates a new timestamped experiment folder. Existing experiments are never resumed or overwritten. The shared trainer also rejects nonempty model output folders. No DMD, gaze, yawn or nod training is offered here. A failed run keeps its log and provenance for inspection.


In [ ]:
from datetime import datetime, timezone
import uuid

if not RUN_TRAINING:
    print("Inspection only. Training skipped; no experiment folder created.")
else:
    assert readiness[TASK]["ready"], readiness[TASK]
    assert not missing_packages, missing_packages
    selected = loaded[TASK]
    import torch
    import torchvision

    assert Path(__import__("src").__file__).resolve() == REPO / "src/__init__.py"
    if DEVICE == "cuda":
        assert torch.cuda.is_available(), (
            "CUDA unavailable: choose a GPU kernel or set DEVICE='cpu'."
        )
    parent = Path("/kaggle/working/experiments") if ON_KAGGLE else REPO / "results/experiments"
    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "_"
        + TASK
        + "_"
        + selected["config"]["model"]
        + "_"
        + uuid.uuid4().hex[:8]
    )
    experiment = parent / run_id
    experiment.mkdir(parents=True, exist_ok=False)
    model_output = experiment / "model"
    # Preserve the exact effective config; no source file or SOURCE_COMMIT is modified.
    effective_config = experiment / "config.yaml"
    effective_config.write_text(yaml.safe_dump(selected["config"]), encoding="utf-8")
    command = [
        sys.executable,
        "-m",
        "src.train.phase1",
        "--data-root",
        str(selected["root"]),
        "--manifest",
        str(selected["manifest"]),
        "--config",
        str(effective_config),
        "--output",
        str(model_output),
        "--splits",
        str(selected["split_path"]),
        "--device",
        DEVICE,
    ]
    if FAST_RUN:
        command.append("--fast")
    provenance = {
        "task": TASK,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_root": str(REPO),
        "source_revision": source_revision,
        "tracked_source_dirty": source_dirty,
        "source_sha256": source_files,
        "manifest_sha256": sha256_file(selected["manifest"]),
        "split_sha256": sha256_file(selected["split_path"]),
        "split_fingerprint": selected["fingerprint"],
        "config": selected["config"],
        "fast": FAST_RUN,
        "device": DEVICE,
        "python": sys.version,
        "command": command,
    }
    (experiment / "run_provenance.json").write_text(
        json.dumps(provenance, indent=2), encoding="utf-8"
    )
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO)
    env["PYTHONUNBUFFERED"] = "1"
    print("Experiment:", experiment)
    with (experiment / "training.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=REPO,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
    (experiment / "run_status.json").write_text(
        json.dumps(
            {"returncode": returncode, "completed_utc": datetime.now(timezone.utc).isoformat()}
        ),
        encoding="utf-8",
    )
    RESULTS_RUN = str(experiment)
    if returncode:
        raise RuntimeError(f"Training failed ({returncode}); inspect {experiment / 'training.log'}")


## 6. View results

Inspect the validation scores and selected checkpoint. The State Farm candidate intentionally produces no test score; final test evaluation follows only after choosing and freezing the recipe using validation.

After training, run step 7 and press Ctrl+S to preserve the model backup inside this local notebook. Saving a notebook without that backup output does not preserve remote model files.


In [ ]:
if RESULTS_RUN:
    result_dir = Path(RESULTS_RUN).expanduser().resolve()
    print("Experiment:", result_dir)
    for relative in ("run_status.json", "model/summary.json", "model/val.json", "model/test.json"):
        path = result_dir / relative
        if path.is_file():
            print(relative)
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2))
    if result_dir.is_dir():
        print("Saved files:")
        for path in sorted(result_dir.rglob("*")):
            if path.is_file():
                print(path.relative_to(result_dir), f"({path.stat().st_size:,} bytes)")
    else:
        print("Folder unavailable in this kernel. Copy or download it first.")
else:
    print("No experiment selected. Inspection-only execution produced no training results.")


## 7. Save a local model backup

After a completed run this cell attaches the experiment ZIP to notebook output. **Press Ctrl+S after this cell finishes.** The readable Python code stays plain text; only binary model artifacts use an encoded notebook attachment. No cloud session can back up to your laptop until the notebook is saved locally.

You can also download the printed ZIP from Kaggle. To restore the embedded copy locally, run `.venv/Scripts/python.exe -m src.train.notebook_backup notebooks/driver_monitoring_training.ipynb`. Preserve these output attachments when saving.


In [ ]:
if RESULTS_RUN:
    from IPython.display import display
    from src.train.notebook_backup import MIME, backup_payload

    payload = backup_payload(RESULTS_RUN)
    display({
        "text/plain": (
            "MODEL BACKUP ATTACHED. Press Ctrl+S now to save it in this local notebook.\n"
            "Experiment: " + payload["experiment"] + "\n"
            "SHA-256: " + payload["sha256"]
        ),
        MIME: payload,
    }, raw=True)
    print("Separate ZIP:", str(Path(RESULTS_RUN)) + "_backup.zip")
else:
    print("No trained experiment selected; backup not created.")
